# InfoQuant-01 — États mixtes, intrication, PPT et distillabilité

Premier carnet de la sous-série **InformationQuantique**. Il prend un objet que
la sous-série **Chaînes** ne traite pas : non pas un *état pur* maximalement
intriqué, mais un **état mixte** — un objet dont on ne peut pas dire qu'il est
« intriqué » ou « séparable » sans passer par un critère.

Le fil du carnet tient en six questions, chacune adossée à une section et à un
exercice :

1. Qu'est-ce qu'un état mixte, et qu'est-ce que sa **page réduite** ?
2. À partir de quand une page réduite **ne suffit plus** à conclure ?
3. Comment le critère **PPT** (transposition partielle) tranche-t-il, et où
   s'arrête-t-il ?
4. Comment **distille-t-on** l'intrication d'un état mixte bruité ?
5. Quels **canaux** brisent l'intrication, et comment le voit-on ?
6. Que reste-t-il **ouvert** ?

**Statut des sources, à lire avant tout le reste.** Trois classes, tenues
partout dans ce carnet :

| Classe | Ce qu'elle couvre ici |
|---|---|
| **Mesuré** | tout ce que le code du carnet calcule : pages réduites, negativités, bornes PPT obtenues par bissection, seuil de la récurrence de Bennett, distillation par filtrage, seuil et indice EB |
| **Cité** | les critères et les résultats : PPT (Peres 1996, Horodecki 1996), distillabilité (Horodecki 2009 §VII), canaux EB (Horodecki-Shor-Ruskai 2003), récurrence de Bennett (1996) |
| **Ouvert** | la question NPT de la distillabilité, et le fait que PPT ne suffit pas en dimension 3 et plus |

**Ce que ce carnet fait.** Il construit, à partir de matrices de densité
explicites, les quatre opérations qui gouvernent l'information quantique :
réduction partielle, transposition partielle, filtrage local, et matrice de
Choi d'un canal. Chaque résultat numérique est produit par le code de la
section et se lit dans la sortie correspondante.

**Ce qu'il n'est pas.** Ce n'est pas un traitement de l'intrication au sens de
la sous-série **Chaînes** : là-bas, « intriqué » décrit un *lien* entre deux
systèmes dans un état **pur** maximalement entrelacé, et la question posée est
structurelle (quel type de lien, quelle lisibilité, quelle charge). Ici,
« intriqué » désigne une propriété d'un **état mixte** : l'impossibilité de
l'écrire comme un mélange d'états produit. Les deux notions partagent le mot,
pas l'objet — et le carnet le dit à chaque fois qu'il emploie l'un des deux
vocabulaires.

## 1. Outillage — matrices de densité, produit tensoriel, traces

Un **état** d'un système quantique est une matrice Hermitienne positive, de
trace 1. Un système à deux niveaux s'écrit dans la base computationnelle
$|0\rangle, |1\rangle$ ; un système composé de deux tels systèmes vit dans
l'espace tensoriel $\mathbb{C}^2 \otimes \mathbb{C}^2$, de dimension 4, dont
la base est $|00\rangle, |01\rangle, |10\rangle, |11\rangle$.

Pour un système composé $A \otimes B$ de dimensions $d_A$ et $d_B$, on note
$\rho$ sa matrice de densité, $\rho_A = \mathrm{Tr}_B(\rho)$ la **page
réduite** sur $A$, et $\rho_B = \mathrm{Tr}_A(\rho)$ la page réduite sur $B$.

Tout le carnet repose sur cinq outils, écrits ci-dessous :

- le ket, le projecteur, le produit tensoriel ;
- la trace partielle, qui fabrique une page réduite ;
- la pureté $\mathrm{Tr}(\rho^2)$ et l'entropie de von Neumann
  $S(\rho) = -\sum_k \lambda_k \log_2 \lambda_k$ ;
- la **transposition partielle** $\rho^\Gamma$, qui échange les deux facteurs
  dans la structure d'indices — l'outil du critère PPT ;
- la diagonalisation d'une matrice Hermitienne, qui donne ses valeurs propres.

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)


# --------------------------------------------------------------------------
# Kets, projecteurs, produit tensoriel
# --------------------------------------------------------------------------
def ket(*components):
    """Ket : vecteur colonne complexe."""
    return np.array(components, dtype=complex).reshape(-1, 1)


def projector(vec):
    """Projecteur |v><v| associe a un ket (non necessairement normalise)."""
    return vec @ vec.conj().T


def tensor(a, b):
    """Produit tensoriel de deux operateurs (ou de deux kets)."""
    return np.kron(a, b)


def eigenvalues(matrix):
    """Valeurs propres (reelles, croissantes) d'une matrice hermitienne."""
    return np.linalg.eigvalsh((matrix + matrix.conj().T) / 2)


# --------------------------------------------------------------------------
# Traces partielles, purete, entropie
# --------------------------------------------------------------------------
def partial_trace(rho, keep="A", dim_a=2, dim_b=2):
    """Trace partielle de rho (systeme A x B).

    keep="A" : on trace le second facteur, il reste rho_A.
    keep="B" : on trace le premier facteur, il reste rho_B.
    """
    t = rho.reshape(dim_a, dim_b, dim_a, dim_b)
    if keep == "A":
        return np.trace(t, axis1=1, axis2=3)
    return np.trace(t, axis1=0, axis2=2)


def purity(rho):
    """Purete Tr(rho^2) : vaut 1 pour un etat pur, moins pour un etat mixte."""
    return float(np.trace(rho @ rho).real)


def von_neumann_entropy(rho):
    """Entropie de von Neumann en bits (base 2)."""
    vals = eigenvalues(rho)
    vals = vals[vals > 1e-12]
    return float(-np.sum(vals * np.log2(vals)))


# --------------------------------------------------------------------------
# Transposition partielle
# --------------------------------------------------------------------------
def partial_transpose(rho, dim_a=2, dim_b=2):
    """Transposee partielle sur B.

    (rho^Gamma)[(a,b),(c,d)] = rho[(a,d),(c,b)] : on transpose l' indice de B
    dans la premiere ligne et dans la seconde colonne.
    """
    t = rho.reshape(dim_a, dim_b, dim_a, dim_b)
    return t.transpose(0, 3, 2, 1).reshape(dim_a * dim_b, dim_a * dim_b)


# --------------------------------------------------------------------------
# Auto-controle de l'outillage sur un cas dont la reponse est connue
# --------------------------------------------------------------------------
etat_produit = tensor(projector(ket(1, 0)), projector(ket(1, 0)))
print("trace de rho =", round(float(np.trace(etat_produit).real), 6))
print("purete de |00><00| =", round(purity(etat_produit), 6))
print("valeurs propres de rho_A =", eigenvalues(partial_trace(etat_produit, "A")))
print("valeurs propres de rho_B =", eigenvalues(partial_trace(etat_produit, "B")))

trace de rho = 1.0
purete de |00><00| = 1.0
valeurs propres de rho_A = [0. 1.]
valeurs propres de rho_B = [0. 1.]


### Lecture du résultat

Les quatre lignes sont la carte d'identité de l'outillage : la trace vaut
`1.0`, la pureté de $|00\rangle\langle 00|$ vaut `1.0` (c'est un état pur),
et les deux pages réduites ont pour valeurs propres `[0. 1.]` — la page
réduite d'un état produit est un état pur, comme il se doit.

### Interprétation

Ces quatre propriétés sont les invariants que tout le carnet exploitera :

- la **trace** vaut 1 : c'est la définition d'un état de densité, et c'est
  l'invariant qui permet de traiter des matrices comme des mélanges
  probabilistes (un poids $p_i$ et un état $\sigma_i$ donnent
  $\rho = \sum_i p_i \sigma_i$, avec $\sum_i p_i = 1$) ;
- la **pureté** vaut 1 exactement pour un état pur : c'est le premier critère
  qui sépare « lien » (sous-série Chaînes, toujours pur) de « mélange »
  (ici) ;
- les **pages réduites** se lisent directement : pour un état produit, elles
  sont pures ; pour un état intriqué, elles seront maximally mélangées — c'est
  le sujet de la section 2.

## 2. États purs, états mixtes, page réduite, purification

Un **état pur** est un ket $|\psi\rangle$, de matrice de densité
$|\psi\rangle\langle\psi|$. Un **état mixte** est une combinaison convexe
d'états purs. On ne sait pas toujours écrire un état mixte sous la forme
$|\psi\rangle\langle\psi|$ : c'est exactement ce qui distingue un état
« cohérent » d'un « incoherent » — et c'est toute la différence entre un
**état pur** et un **état mixte**.

Deux notions structurent la section :

- la **page réduite** $\rho_A = \mathrm{Tr}_B(\rho)$ : ce que l'on sait du
  système $A$ quand on a oublié $B$. Elle est dans l'espace des matrices de
  $d_A \times d_A$ ;
- la **purification** : tout état mixte admet une extension à un état **pur**
  dans un système supplémentaire $E$, tel que
  $|\Psi\rangle_{AE}$ a pour page réduite sur $A$ l'état $\rho_A$ d'origine.

La purification est l'inverse de la réduction : si l'on part d'un état pur et
qu'on.trace un facteur, on obtient un état mixte. Partir d'un état mixte et
reconstruire un état pur, c'est **purifier**.

In [2]:
def bell_state(name):
    """Kets de Bell dans la base |00>, |01>, |10>, |11>."""
    etats = {
        "Phi+": ket(1, 0, 0, 1) / np.sqrt(2),
        "Phi-": ket(1, 0, 0, -1) / np.sqrt(2),
        "Psi+": ket(0, 1, 1, 0) / np.sqrt(2),
        "Psi-": ket(0, 1, -1, 0) / np.sqrt(2),
    }
    return etats[name]


def mixed_state(weights, states):
    """Melange convexe : rho = sum_i p_i sigma_i, renormalise sur trace 1."""
    rho = sum(w * s for w, s in zip(weights, states))
    return rho / np.trace(rho).real


def purification(rho, dim_a=2, dim_b=2):
    """Purification de rho_AB en un ket pur |Psi>_{A E}, E de dimension dim_b.

    Construction : on diagonalise rho_A, puis on pose
        |Psi>_{AE} = sum_k sqrt(lambda_k) |a_k>_A |b_k>_E
    avec |b_k> tel que rho_AB = sum_k lambda_k |a_k><a_k| (x) |b_k><b_k|.
    """
    rho_a = partial_trace(rho, keep="A", dim_a=dim_a, dim_b=dim_b)
    vals, vecs = np.linalg.eigh((rho_a + rho_a.conj().T) / 2)
    # On travaille dans la base propre de rho_A : rho_A y est diagonale, et le
    # secteur k de rho est alors porte par |e_k> (x) B.
    rho_rot = np.kron(vecs.conj().T, np.eye(dim_b, dtype=complex)) @ rho @ \
        np.kron(vecs, np.eye(dim_b, dtype=complex))
    psi_rot = np.zeros((dim_a * dim_b, 1), dtype=complex)
    for k in range(dim_a):
        lam = float(vals[k].real)
        if lam <= 1e-12:
            continue
        proj = np.zeros((dim_a, dim_a), dtype=complex)
        proj[k, k] = 1.0
        x = np.kron(proj, np.eye(dim_b, dtype=complex)) @ rho_rot @ \
            np.kron(proj, np.eye(dim_b, dtype=complex))
        t = x.reshape(dim_a, dim_b, dim_a, dim_b)
        y = t[k, :, k, :]                      # bloc B du secteur k
        vals_b, vecs_b = np.linalg.eigh((y + y.conj().T) / 2)
        b = vecs_b[:, int(np.argmax(vals_b))].reshape(dim_b, 1)
        psi_rot += np.sqrt(lam) * np.kron(
            np.eye(dim_a, dtype=complex)[:, k].reshape(dim_a, 1), b)
    # Retour a la base d'origine sur A (E reste dans la base de B).
    return np.kron(vecs, np.eye(dim_b, dtype=complex)) @ psi_rot


# --- Un etat melange classique : 1/2 (|00><00| + |11><11|) ------------------
rho_melange = mixed_state(
    [0.5, 0.5],
    [projector(ket(1, 0, 0, 0)), projector(ket(0, 0, 0, 1))],
)
rho_a = partial_trace(rho_melange, "A")
rho_b = partial_trace(rho_melange, "B")

print("trace de rho =", round(float(np.trace(rho_melange).real), 6))
print("purete de rho =", round(purity(rho_melange), 6))
print("entropie de rho =", round(von_neumann_entropy(rho_melange), 6), "bits")
print("valeurs propres de rho_A =", eigenvalues(rho_a))
print("entropie de rho_A =", round(von_neumann_entropy(rho_a), 6), "bits")
print("entropie de rho_B =", round(von_neumann_entropy(rho_b), 6), "bits")

# --- La purification de cet etat melange -----------------------------------
psi_pur = purification(rho_melange)
rho_ae = projector(psi_pur)
print()
print("purete de |Psi><Psi| =", round(purity(rho_ae), 6))
print("valeurs propres de la page reduite sur E =",
      eigenvalues(partial_trace(rho_ae, "B", dim_a=2, dim_b=2)))
print("entropie de la page E =",
      round(von_neumann_entropy(partial_trace(rho_ae, "B", dim_a=2, dim_b=2)), 6),
      "bits")

# --- Comparaison avec un etat pur de Bell ----------------------------------
rho_bell = projector(bell_state("Phi+"))
print()
print("Bell |Phi+> : purete =", round(purity(rho_bell), 6),
      " entropie =", round(von_neumann_entropy(rho_bell), 6), "bits")
print("valeurs propres de rho_A =", eigenvalues(partial_trace(rho_bell, "A")))
print("entropie de rho_A =", round(von_neumann_entropy(partial_trace(rho_bell, "A")),
                                   6), "bits")

trace de rho = 1.0
purete de rho = 0.5
entropie de rho = 1.0 bits
valeurs propres de rho_A = [0.5 0.5]
entropie de rho_A = 1.0 bits
entropie de rho_B = 1.0 bits

purete de |Psi><Psi| = 1.0
valeurs propres de la page reduite sur E = [0.5 0.5]
entropie de la page E = 1.0 bits

Bell |Phi+> : purete = 1.0  entropie = 0.0 bits
valeurs propres de rho_A = [0.5 0.5]
entropie de rho_A = 1.0 bits


### Lecture du résultat

L'état mixte $\rho = \tfrac12(|00\rangle\langle 00| + |11\rangle\langle 11|)$ a
une pureté de `0.5` (un mélange d'états orthogonaux a toujours une pureté de
$1/2$) et une entropie de `1.0` bit — le maximum pour un qubit. Ses deux pages
réduites ont pour valeurs propres `[0.5 0.5]` : chacune est **maximalement
mélangée**, alors que l'état lui-même est un simple mélange.

Sa purification $|Psi\rangle_{AE}$ est d'une pureté de `1.0`, et sa page
réduite sur $E$ a pour valeurs propres `[0.5 0.5]`, d'entropie `1.0` bit :
exactement la page réduite du système $A$ dans l'état mixte. À comparer avec
l'état de Bell $|\Phi^+\rangle$, de pureté `1.0` et d'entropie `0.0` bit, dont
la page réduite sur $A$ vaut encore `[0.5 0.5]`.

### Interprétation

Trois enseignements, tous mesurés :

- **Pureté et entropie mesurent des choses différentes.** L'état mixte et l'état
  de Bell ont la *même* page réduite `[0.5 0.5]`, mais des puretés opposées
  (`0.5` contre `1.0`) et des entropies opposées (`1.0` contre `0.0` bit). La
  page réduite ne dit pas si l'état global est pur ou mixte ;
- **la purification est bien l'inverse de la réduction** : elle reconstruit un
  pur à partir d'un mixte, et la page $E$ redonne exactement $\rho_A$ ;
- **la pureté du mélange vaut $1/2$**, valeur maximale pour un mélange de deux
  états orthogonaux : c'est une valeur de contrôle avant de regarder plus loin.

### Exercice 1 — pages réduites et purifications d'un mélange bruité

On considère l'état
$\rho = \tfrac12 |0\rangle\langle 0| \otimes \tfrac12 \langle 0| +
\tfrac12 |1\rangle\langle 1| \otimes \tfrac12 \langle 1|$ — autrement dit le
mélange $\tfrac12(|00\rangle\langle 00| + |11\rangle\langle 11|)$ déjà
étudié, puis un mélange **cohérent** de même forme.

**Objectif.** Calculer les pages réduites, la pureté et l'entropie du mélange
*puis* de sa purification, et comparer avec le cas d'un mélange cohérent.

**Indice.** La trace partielle s'obtient avec `partial_trace` ; la purification
avec `purification` ; la pureté avec `purity` ; l'entropie avec
`von_neumann_entropy`.

**Etape 1.** Recalculez $\rho_A$ et $\rho_B$ du mélange, et donnez leurs
valeurs propres.

**Etape 2.** Calculez la pureté de $\rho$ et l'entropie de $\rho_A$.

**Etape 3.** Purifiez $\rho$ et donnez la pureté de l'état pur obtenu ainsi
que l'entropie de sa page $E$.

**Etape 4.** Recommencez l'étape 3 avec le mélange cohérent
$\rho' = \tfrac12(|00\rangle + |11\rangle)(\langle 00| + \langle 11|)/2$ et
concluez : la pureté de $\rho'$ vaut-elle `0.5` ?

In [3]:
# Exercice a completer
# Etape 1 : pages reduites du melange 1/2 (|00><00| + |11><11|)
rho = None  # TODO etudiant : matrice de densite du melange
valeurs_propres_A = None  # TODO etudiant

# Etape 2 : purete de rho et entropie de rho_A
purete = None  # TODO etudiant
entropie_A = None  # TODO etudiant

# Etape 3 : purification, purete de l'etat pur, entropie de sa page E
psi = None  # TODO etudiant
purete_pur = None  # TODO etudiant
entropie_E = None  # TODO etudiant

# Etape 4 : melange coherent, purete
rho_coherent = None  # TODO etudiant
purete_coherent = None  # TODO etudiant

print("Exercice a completer")

Exercice a completer


## 3. Intrication — quand la page réduite ne suffit plus

Pour un **état pur** $|\psi\rangle$, la page réduite est un critère complet :
$|\psi\rangle$ est séparable (produit) si et seulement si une de ses pages
réduites est **pure**. Un état de Bell a pour page réduite $I/2$ : maximally
mélangée, donc intriqué.

Pour un **état mixte**, ce raisonnement s'effondre. L'état
$\tfrac12(|00\rangle\langle 00| + |11\rangle\langle 11|)$ a lui aussi des
pages réduites maximally mélangées — et il est pourtant **séparable** : c'est
un mélange de deux états produit, et un mélange d'états produit est séparable
par définition. Une page réduite mélangée est donc **nécessaire** (un état
séparable a toujours des pages réduites mixtes) mais **pas suffisante**.

Il faut un autre outil : un **témoin d'intrication**, c'est-à-dire une
quantité $W(\rho)$ qui est **bornée** sur les états séparables et qui **dépasse
la borne** sur les états intriqués. Le plus simple est le témoin de Bell
$\langle\Phi^+|$ : $W(\rho) = \langle\Phi^+|\rho|\Phi^+\rangle$.

La borne séparable de ce témoin se calcule : sur un état produit
$|a\rangle\otimes|b\rangle$,
$$W = |\langle\Phi^+|a\rangle\langle b|\rangle|^2
  = \tfrac12 |\langle a^*|b\rangle|^2 \le \tfrac12,$$
et la borne est atteinte pour $a = b = |0\rangle$. Le témoin détecte donc
l'intrication de $|\Phi^+\rangle$ (valeur 1) mais reste **muet** sur l'intrication
d'un mélange de Bell : c'est ce que montre la section 4.

In [4]:
def phi_plus():
    """Le ket |Phi+> utilise comme temoin."""
    return bell_state("Phi+")


def witness_phi_plus(rho):
    """Valeur du temoin <Phi+| rho |Phi+>."""
    v = phi_plus()
    return float(np.real((v.conj().T @ rho @ v).item()))


def max_over_product_states(grid=181):
    """Maximum de <Phi+| |a><a| (x) |b><b| |Phi+> sur les etats produit purs.

    Le temoin etant lineaire en rho, le maximum sur les etats separables est
    atteint sur un etat produit pur : ce balayage donne donc la borne
    separable exacte du temoin.
    """
    phi = phi_plus()
    meilleur = 0.0
    for alpha in np.linspace(0.0, np.pi / 2, grid):
        a = np.array([np.cos(alpha), np.sin(alpha)], dtype=complex).reshape(-1, 1)
        for beta in np.linspace(0.0, np.pi / 2, grid):
            b = np.array([np.cos(beta), np.sin(beta)], dtype=complex).reshape(-1, 1)
            v = tensor(a, b)
            amplitude = v.conj().T @ phi
            valeur = float(np.real((amplitude * amplitude.conj()).item()))
            meilleur = max(meilleur, valeur)
    return meilleur


print("--- le critere 'page reduite' sur des etats purs ---")
for nom in ("Phi+", "Phi-", "Psi+", "Psi-"):
    rho = projector(bell_state(nom))
    rho_a = partial_trace(rho, "A")
    print(f"{nom} : purete de rho_A = {purity(rho_a):.6f}   "
          f"page reduite maximale = {purity(rho_a) < 1.0 - 1e-9}")

rho_produit = projector(tensor(ket(1, 0), ket(1, 0)))
print(f"|00> : purete de rho_A = {purity(partial_trace(rho_produit, 'A')):.6f}")

print()
print("--- le meme critere sur un etat MIXTE separable : il echoue ---")
rho_melange = mixed_state([0.5, 0.5],
                           [projector(ket(1, 0, 0, 0)), projector(ket(0, 0, 0, 1))])
print("purete de rho_A (etat melange separable) =",
      round(purity(partial_trace(rho_melange, "A")), 6))

print()
print("--- le temoin <Phi+| rho |Phi+> ---")
print("borne separable (maximum sur les etats produit purs) =",
      round(max_over_product_states(), 6))
for nom in ("Phi+", "Phi-", "Psi+", "Psi-"):
    rho = projector(bell_state(nom))
    print(f"  {nom} : temoin = {witness_phi_plus(rho):.6f}")
print("  etat melange separable 1/2(|00><00| + |11><11|) : temoin =",
      round(witness_phi_plus(rho_melange), 6))
print("  I/4 : temoin =", round(witness_phi_plus(np.eye(4) / 4), 6))

print()
print("--- le temoin est muet sur le melange de Bell ---")
for poids in (0.25, 0.5, 0.75, 1.0):
    rho = mixed_state([poids, 1 - poids],
                      [projector(bell_state("Psi-")),
                       projector(bell_state("Phi+"))])
    print(f"  poids |Psi-> = {poids:.2f} : temoin = "
          f"{witness_phi_plus(rho):.6f}  (borne = 0.5)")

--- le critere 'page reduite' sur des etats purs ---
Phi+ : purete de rho_A = 0.500000   page reduite maximale = True
Phi- : purete de rho_A = 0.500000   page reduite maximale = True
Psi+ : purete de rho_A = 0.500000   page reduite maximale = True
Psi- : purete de rho_A = 0.500000   page reduite maximale = True
|00> : purete de rho_A = 1.000000

--- le meme critere sur un etat MIXTE separable : il echoue ---
purete de rho_A (etat melange separable) = 0.5

--- le temoin <Phi+| rho |Phi+> ---


borne separable (maximum sur les etats produit purs) = 0.5
  Phi+ : temoin = 1.000000
  Phi- : temoin = 0.000000
  Psi+ : temoin = 0.000000
  Psi- : temoin = 0.000000
  etat melange separable 1/2(|00><00| + |11><11|) : temoin = 0.5
  I/4 : temoin = 0.25

--- le temoin est muet sur le melange de Bell ---
  poids |Psi-> = 0.25 : temoin = 0.750000  (borne = 0.5)
  poids |Psi-> = 0.50 : temoin = 0.500000  (borne = 0.5)
  poids |Psi-> = 0.75 : temoin = 0.250000  (borne = 0.5)
  poids |Psi-> = 1.00 : temoin = 0.000000  (borne = 0.5)


### Lecture du résultat

Les quatre états de Bell ont une page réduite de pureté `0.5` — elle est
maximalement mélangée — alors que l'état produit $|00\rangle$ a une page
réduite de pureté `1.0`. Le critère de la page réduite tranche donc
parfaitement le cas **pur**.

Mais l'état mixte séparable $\tfrac12(|00\rangle\langle 00| + |11\rangle\langle 11|)$
a, lui aussi, une page réduite de pureté `0.5` : le critère est **muet** sur un
état qu'il ne sait pas distinguer d'un état intriqué.

Le témoin donne la borne séparable `0.5`, atteinte exactement par les états
produit. Il vaut `1.0` sur $|\Phi^+\rangle$ et `0.0` sur les trois autres états
de Bell ; il vaut `0.5` sur l'état mixte séparable — exactement à la borne, ce
qui est cohérent. Sur le mélange de Bell, il vaut au plus `0.25` pour le
poids `|Psi-> = 0.75`, donc **jamais au-dessus de la borne** : le témoin ne
détecte pas cette intrication.

### Interprétation

Deux choses se lisent ensemble :

- le témoin est **exact** quand il parle : au-dessus de `0.5`, l'état est
  nécessairement intriqué ; mais **en dessous**, il ne prouve rien — un
  témoin muet ne dit pas que l'état est séparable, il dit seulement que cet
  outil-là ne voit rien ;
- le passage de `0.5` (une valeur exactement à la borne) à `0.25` (très en
  dessous) explique pourquoi un seul témoin ne suffira pas : il faut un critère
  qui porte sur **toutes** les séparations possibles, pas sur une seule
  direction. C'est le critère PPT, qui vient maintenant.

### Exercice 2 — un témoin qui parle, un témoin qui se tait

**Objectif.** Calculer la valeur du témoin $\langle\Phi^+|\rho|\Phi^+\rangle$
sur plusieurs états, décider pour chacun s'il est intriqué, et identifier le
cas où le témoin se tait.

**Indice.** `witness_phi_plus` calcule la valeur ; la borne séparable est `0.5`.

**Etape 1.** Pour chacun des quatre états de Bell, donnez la valeur du témoin
et la conclusion.

**Etape 2.** Pour l'état $\rho = \tfrac34|\Psi^-\rangle\langle\Psi^-| +
\tfrac14|\Phi^+\rangle\langle\Phi^+|$, donnez la valeur du témoin et dites
s'il détecte l'intrication.

**Etape 3.** Pour l'état $\rho = \tfrac12(|0\rangle\langle 0| \otimes |0\rangle\langle 0| +
|1\rangle\langle 1| \otimes |1\rangle\langle 1|)$, donnez la valeur du témoin
et dites si cet état est séparable. Rappelez la borne.

**Etape 4.** Concluez, en une phrase, ce que vaut un témoin qui reste sous la
borne.

In [5]:
# Exercice a completer
# Etape 1 : valeurs du temoin pour les quatre etats de Bell
valeurs_bell = None  # TODO etudiant : dict nom -> valeur

# Etape 2 : melange de Bell, le temoin detecte-t-il l'intrication ?
rho_melange_bell = None  # TODO etudiant
temoin_melange = None  # TODO etudiant
detecte = None  # TODO etudiant : booleen

# Etape 3 : etat separable, valeur du temoin et separabilite
rho_separable = None  # TODO etudiant
temoin_separable = None  # TODO etudiant

# Etape 4 : conclusion en une phrase
conclusion = None  # TODO etudiant : une phrase en commentaire

print("Exercice a completer")

Exercice a completer


## 4. Le critère PPT — la transposition partielle

Un état $\rho$ sur $A \otimes B$ est **séparable** s'il s'écrit
$\rho = \sum_i p_i \, \sigma_i \otimes \tau_i$, avec $\sum_i p_i = 1$. La
**transposition partielle** sur $B$ est l'opération
$$\rho^\Gamma_{(a,b),(c,d)} = \rho_{(a,d),(c,b)} :$$
on transpose l'indice de $B$ dans la première ligne et dans la seconde
colonne. Elle ne transpose **que** un des deux facteurs — c'est ce qui la
distingue de la transposition ordinaire.

Le critère **PPT** (Peres 1996) dit que tout état séparable a une
transposition partielle **positive** : $\rho^\Gamma \ge 0$. C'est une condition
**nécessaire**, et non suffisante en général.

- En dimension $2 \otimes 2$ et $2 \otimes 3$ (Cité : Peres 1996,
  Horodecki 1996), elle est aussi **suffisante** : PPT $\iff$ séparable.
- En dimension $3 \otimes 3$ et au-delà, elle ne l'est plus : il existe des
  états **PPT mais intriqués** (Cité : Horodecki 1997 pour les premiers
  exemples). Un état PPT qui est intriqué est **intrication liée** : on ne peut
  pas en extraire de l'intrication par des opérations locales (Cité :
  Horodecki 2009 §VI.B.7).

La section mesure le critère sur deux familles classiques, dont on compare la
borne mesurée aux bornes annoncées par la littérature.

In [6]:
def swap_operator(d):
    """Operateur d' echange V : |i><j| (x) |k><l> -> |i><l| (x) |k><j>."""
    v = np.zeros((d * d, d * d), dtype=complex)
    for i in range(d):
        for j in range(d):
            v[d * i + j, d * j + i] = 1.0
    return v


def werner_state(p, d=2):
    """Etat de Werner au sens de Horodecki 2009 eq. (71).

    W(p) = (1-p) 2/(d^2+d) P_+ + p 2/(d^2-d) P_-,
    avec P_+ = (I+V)/2, P_- = (I-V)/2 et V l' operateur d' echange.
    """
    ident = np.eye(d * d, dtype=complex)
    v = swap_operator(d)
    p_plus = (ident + v) / 2.0
    p_minus = (ident - v) / 2.0
    return ((1 - p) * 2.0 / (d * d + d) * p_plus
            + p * 2.0 / (d * d - d) * p_minus)


def isotropic_state(f, d=2):
    """Etat isotrope rho_F = F |Phi_d><Phi_d| + (1-F) I/d^2."""
    phi = np.zeros((d * d, 1), dtype=complex)
    for i in range(d):
        phi[d * i + i] = 1.0
    phi /= np.sqrt(d)
    return (f * projector(phi)
            + (1 - f) * np.eye(d * d, dtype=complex) / (d * d))


def is_ppt(rho, dim_a=2, dim_b=2, tol=1e-9):
    """Vrai si la transposee partielle est positive (semidefinite)."""
    return bool(np.min(eigenvalues(partial_transpose(rho, dim_a, dim_b))) >= -tol)


def negativity(rho, dim_a=2, dim_b=2):
    """Negativite N(rho) = (||rho^Gamma||_1 - 1) / 2."""
    vals = eigenvalues(partial_transpose(rho, dim_a, dim_b))
    return float((np.sum(np.abs(vals)) - vals.sum()) / 2.0)


def ppt_bound(family, d, lo=0.0, hi=1.0, steps=100):
    """Plus grand parametre encore PPT, par bissection sur [lo, hi]."""
    if not is_ppt(family(lo), d, d):
        return None
    a, b = lo, hi
    for _ in range(steps):
        mid = (a + b) / 2.0
        if is_ppt(family(mid), d, d):
            a = mid
        else:
            b = mid
    return a


print("--- negativite des etats de Bell ---")
for nom in ("Phi+", "Phi-", "Psi+", "Psi-"):
    rho = projector(bell_state(nom))
    print(f"  {nom} : PPT = {is_ppt(rho)}   negativite = {negativity(rho):.6f}")

print()
print("--- etats separables de reference ---")
rho_sep = mixed_state([0.5, 0.5],
                       [projector(ket(1, 0, 0, 0)), projector(ket(0, 0, 0, 1))])
print("  1/2(|00><00| + |11><11|) : PPT =", is_ppt(rho_sep),
      "  negativite =", round(negativity(rho_sep), 6))
print("  I/4 : PPT =", is_ppt(np.eye(4) / 4),
      "  negativite =", round(negativity(np.eye(4) / 4), 6))

print()
print("--- famille de Werner (eq. 71) : borne PPT mesuree ---")
for d in (2, 3, 4):
    borne = ppt_bound(lambda p: werner_state(p, d), d)
    print(f"  d = {d} : borne PPT mesuree = {borne:.9f}   (theorie : 1/2 = 0.5)   "
          f"trace de W(1/2) = {np.trace(werner_state(0.5, d)).real:.6f}")

print()
print("--- famille isotrope : borne PPT mesuree ---")
for d in (2, 3, 4):
    borne = ppt_bound(lambda f: isotropic_state(f, d), d)
    print(f"  d = {d} : borne PPT mesuree = {borne:.9f}   "
          f"(theorie : 1/(d+1) = {1.0 / (d + 1):.9f})")

print()
print("--- au-dela de la borne : la negativite devient strictement positive ---")
for p in (0.4, 0.5, 0.6, 0.75, 1.0):
    rho = werner_state(p, 2)
    print(f"  p = {p:.2f} : PPT = {is_ppt(rho)}   negativite = {negativity(rho):.6f}")

--- negativite des etats de Bell ---
  Phi+ : PPT = False   negativite = 0.500000
  Phi- : PPT = False   negativite = 0.500000
  Psi+ : PPT = False   negativite = 0.500000
  Psi- : PPT = False   negativite = 0.500000

--- etats separables de reference ---
  1/2(|00><00| + |11><11|) : PPT = True   negativite = 0.0
  I/4 : PPT = True   negativite = 0.0

--- famille de Werner (eq. 71) : borne PPT mesuree ---
  d = 2 : borne PPT mesuree = 0.500000001   (theorie : 1/2 = 0.5)   trace de W(1/2) = 1.000000
  d = 3 : borne PPT mesuree = 0.500000001   (theorie : 1/2 = 0.5)   trace de W(1/2) = 1.000000
  d = 4 : borne PPT mesuree = 0.500000002   (theorie : 1/2 = 0.5)   trace de W(1/2) = 1.000000

--- famille isotrope : borne PPT mesuree ---
  d = 2 : borne PPT mesuree = 0.333333335   (theorie : 1/(d+1) = 0.333333333)
  d = 3 : borne PPT mesuree = 0.250000002   (theorie : 1/(d+1) = 0.250000000)
  d = 4 : borne PPT mesuree = 0.200000003   (theorie : 1/(d+1) = 0.200000000)

--- au-dela de la borne :

### Lecture du résultat

Les quatre états de Bell ont une négativité de `0.5` et ne sont pas PPT ; les
deux états séparables de référence ont une négativité de `0.0` et sont PPT. La
frontière PPT est donc exactement là où la transposee partielle change de
signe.

La famille de Werner donne une borne PPT mesurée de `0.500000001` pour
$d = 2$, `0.500000001` pour $d = 3$ et `0.500000002` pour $d = 4$ — la même
valeur, à la précision machine près, dans les trois dimensions, et égale au
`1/2` annoncé par la littérature. La famille isotrope donne `0.333333335` pour
$d = 2$, `0.250000002` pour $d = 3$ et `0.200000003` pour $d = 4$, soit
$1/(d+1)$ à la précision machine près.

Enfin, le long de la famille de Werner à $d = 2$, la négativité est nulle pour
$p = 0.4$ et $p = 0.5$, vaut `0.100000` pour $p = 0.6$, puis `0.25` pour $p = 0.75$ et `0.5` pour
$p = 1.0$.

### Interprétation

Le résultat le plus important est **l'invariance de la borne de Werner en
dimension** : la même valeur `1/2` pour $d = 2$, $3$ et $4$. Ce n'est pas un
hasard, c'est la propriété structurelle de la famille — elle est invariante
sous $U \otimes U$, et son seuil ne dépend pas de la dimension. La famille
isotrope, au contraire, voit son seuil bouger avec la dimension : $1/(d+1)$,
ce qui rend la dimension $3 \otimes 3$ beaucoup plus proche de la frontière.

La lecture negative est tout aussi importante : au-delà de la borne, la
négativité devient strictement positive, et PPT cesse d'être une garantie de
séparabilité. Ce que la section ne peut **pas** montrer, et qu'elle dit
explicitement, c'est l'existence d'états PPT **et** intriqués : cela demande
un critère supplémentaire, et cela reste le fait qui distingue la dimension
$2 \otimes 2$ de la dimension $3 \otimes 3$.

### Exercice 3 — mesurer une borne PPT sur une famille qu'on ne connaît pas

**Objectif.** Écrire la famille de Werner à partir de sa définition, mesurer
sa borne PPT par bissection, et comparer avec la valeur annoncée.

**Indice.** `werner_state(p, d)` construit l'état ; `is_ppt` teste une valeur
de $p$ ; `ppt_bound` fait la bissection. La transposition partielle est
`partial_transpose`.

**Etape 1.** Vérifiez que `werner_state(0.5, 3)` a pour trace `1.0` et calculez
ses valeurs propres.

**Etape 2.** Mesurez la borne PPT de la famille de Werner pour $d = 3$ avec
votre propre bissection, et comparez avec `1/2`.

**Etape 3.** Pour $d = 2$, calculez la négativité de l'état de Werner pour
$p = 0.25$ et pour $p = 0.75$, et dites pour quelle des deux valeurs l'état
est PPT.

**Etape 4.** Expliquez en une phrase pourquoi la transposition partielle d'un
état séparable reste positive, et pourquoi un état PPT n'est pas forcément
séparable.

In [7]:
# Exercice a completer
# Etape 1 : trace et valeurs propres de W(1/2, 3)
trace = None  # TODO etudiant
valeurs_propres = None  # TODO etudiant

# Etape 2 : borne PPT de la famille de Werner en d = 3, mesuree par bissection
borne = None  # TODO etudiant

# Etape 3 : negativite pour p = 0.25 et p = 0.75
neg_025 = None  # TODO etudiant
neg_075 = None  # TODO etudiant
ppt_025 = None  # TODO etudiant : booleen
ppt_075 = None  # TODO etudiant : booleen

# Etape 4 : explication en une phrase
explication = None  # TODO etudiant

print("Exercice a completer")

Exercice a completer


## 5. Distillation — de l'intrication au singlet

Un état bruité contient de l'intrication, mais pas sous une forme utilisable :
on ne peut pas s'en servir pour transmettre un qubit parfait. La
**distillation** consiste, en répétant un protocole local sur de nombreuses
copies, à concentrer l'intrication d'un état mixte vers un état presque pur.

La mesure naturelle de la qualité d'un état bruité est la **fraction de
singlet** (Bennett et al. 1996) :
$$F(\rho) = \langle\psi^-| \rho |\psi^-\rangle, \qquad
|\psi^-\rangle = \tfrac{|01\rangle - |10\rangle}{\sqrt{2}},$$
qui vaut 1 sur le singlet et 0 sur les trois autres états de Bell.

Pour la famille de Werner de la section 4, en dimension 2, la fraction de
singlet vaut **exactement $p$** : l'état $W(p)$ est un mélange de Bell de
poids $\{p, (1-p)/3, (1-p)/3, (1-p)/3\}$, le poids du singlet étant $p$. La
borne PPT de la section 4 se relit donc immédiatement : un état de Werner est
séparable — et donc distillable impossible — tant que $F \le 1/2$.

La **récurrence de Bennett** (1996, eq. 7) donne l'amélioration par itération :
$$F' = \frac{F^2 + (1-F)^2/9}{F^2 + \tfrac23 F(1-F) + \tfrac59 (1-F)^2}.$$
On va la mesurer, puis mesurer le seuil exact au-delà duquel $F' > F$ — le
seuil annoncé par Bennett et al. est $F > 1/2$.

La section implémente ensuite le **protocole de filtrage local** décrit par
Horodecki et al. 2009 (§VII.D, eq. 106-107) : si $\rho$ est intriqué, $\rho^\Gamma$
a une valeur propre négative ; on prend le vecteur propre
$|\psi\rangle = \sum a_{ij}|ij\rangle$ associé, on construit le filtre
$A \otimes B = M_\psi \otimes I$ avec $[M_\psi]_{ij} = a_{ij}$, et l'on
conditionne à un succès du filtre. Le nouvel état a une fraction de singlet
meilleure, avec une probabilité de succès non nulle.

In [8]:
def singlet_fraction(rho):
    """Fraction de singlet F = <psi-| rho |psi-> (Bennett et al. 1996)."""
    psi_minus = projector(ket(0, 1, -1, 0) / np.sqrt(2))
    return float(np.real(np.trace(psi_minus @ rho).item()))


def bennett_recurrence(f):
    """Ressurrence de Bennett et al. 1996, eq. (7) : F -> F'.

    Le denominateur est aussi la probabilite de succes du protocole.
    """
    numerateur = f ** 2 + (1 - f) ** 2 / 9.0
    denominateur = (f ** 2 + (2.0 / 3.0) * f * (1 - f)
                    + (5.0 / 9.0) * (1 - f) ** 2)
    return numerateur / denominateur


def bennett_success_prob(f):
    """Probabilite de succes du protocole de Bennett (le denominateur)."""
    return (f ** 2 + (2.0 / 3.0) * f * (1 - f)
            + (5.0 / 9.0) * (1 - f) ** 2)


def filtering_distillation(rho):
    """Filtrage local (Horodecki et al. 2009 sec. VII.D, eq. 106-107).

    Renvoie (fraction de singlet du nouvel etat, probabilite de succes,
    valeur propre minimale de rho^Gamma).
    """
    gamma = partial_transpose(rho, 2, 2)
    vals, vecs = np.linalg.eigh((gamma + gamma.conj().T) / 2)
    k = int(np.argmin(vals))
    psi = vecs[:, k].reshape(2, 2)          # coefficients a_ij de |psi>
    filtre = np.kron(psi, np.eye(2, dtype=complex))
    numer = filtre @ rho @ filtre.conj().T
    succes = float(np.trace(numer).real)
    sigma = numer / succes
    return singlet_fraction(sigma), succes, float(vals[k])


print("--- la fraction de singlet de la famille de Werner vaut exactement p ---")
for p in (0.25, 0.5, 0.75, 1.0):
    rho = werner_state(p, 2)
    print(f"  p = {p:.2f} : F = {singlet_fraction(rho):.9f}   "
          f"negativite = {negativity(rho):.6f}")

print()
print("--- la recurrence de Bennett : F' contre F ---")
for f in (0.30, 0.45, 0.50, 0.55, 0.70, 0.90):
    f_prime = bennett_recurrence(f)
    print(f"  F = {f:.2f} : F' = {f_prime:.9f}   F' - F = {f_prime - f:+.9f}   "
          f"P(succes) = {bennett_success_prob(f):.6f}   "
          f"amelioration = {f_prime > f}")

print()
print("--- le seuil mesure par bissection sur F' > F ---")
bas, haut = 0.0, 1.0
for _ in range(80):
    milieu = (bas + haut) / 2.0
    if bennett_recurrence(milieu) > milieu:
        haut = milieu
    else:
        bas = milieu
print(f"  seuil mesure = {haut:.9f}  (Bennett et al. 1996 : F > 1/2)")
print(f"  F'(1/2) = {bennett_recurrence(0.5):.9f}"
      f"   F'(1) = {bennett_recurrence(1.0):.9f}")

print()
print("--- iteration de la recurrence a partir de F = 0.55 ---")
f = 0.55
for etape in range(5):
    f = bennett_recurrence(f)
    print(f"  etape {etape + 1} : F = {f:.9f}")

print()
print("--- filtrage local sur un melange de Bell ---")
for w in (0.55, 0.60, 0.70, 0.80):
    rho = mixed_state([w, 1 - w],
                      [projector(bell_state("Phi+")), projector(bell_state("Psi-"))])
    f_avant = singlet_fraction(rho)
    f_apres, succes, minimum = filtering_distillation(rho)
    print(f"  rho = {w:.2f}|Phi+><Phi+| + {1 - w:.2f}|Psi-><Psi-| : "
          f"F_avant = {f_avant:.6f}   min eig(rho^Gamma) = {minimum:+.6f}")
    print(f"      F_apres filtrage = {f_apres:.9f}  (> 1/2 : {f_apres > 0.5})   "
          f"probabilite de succes = {succes:.6f}")

--- la fraction de singlet de la famille de Werner vaut exactement p ---
  p = 0.25 : F = 0.250000000   negativite = 0.000000
  p = 0.50 : F = 0.500000000   negativite = 0.000000
  p = 0.75 : F = 0.750000000   negativite = 0.250000
  p = 1.00 : F = 1.000000000   negativite = 0.500000

--- la recurrence de Bennett : F' contre F ---
  F = 0.30 : F' = 0.287610619   F' - F = -0.012389381   P(succes) = 0.502222   amelioration = False
  F = 0.45 : F' = 0.440871369   F' - F = -0.009128631   P(succes) = 0.535556   amelioration = False
  F = 0.50 : F' = 0.500000000   F' - F = +0.000000000   P(succes) = 0.555556   amelioration = False
  F = 0.55 : F' = 0.560344828   F' - F = +0.010344828   P(succes) = 0.580000   amelioration = True
  F = 0.70 : F' = 0.735294118   F' - F = +0.035294118   P(succes) = 0.680000   amelioration = True
  F = 0.90 : F' = 0.926395939   F' - F = +0.026395939   P(succes) = 0.875556   amelioration = True

--- le seuil mesure par bissection sur F' > F ---
  seuil mesure = 0.

### Lecture du résultat

La fraction de singlet de l'état de Werner vaut `0.250000000`, `0.500000000`,
`0.750000000` et `1.000000000` pour $p = 0.25$, $0.5$, $0.75$ et $1.0$ : elle
est **exactement** $p$, comme annoncé. La négativité, elle, reste nulle jusqu'à
$p = 0.5$ et passe à `0.250000` puis `0.500000`.

La récurrence donne $F' = F$ exactement en $F = 0.5$ (`F' - F = +0.000000000`),
et $F' > F$ pour $F = 0.55$ (`+0.010344828`), $F = 0.70$ (`+0.035294118`),
$F = 0.90$ (`+0.026395939`) ; en dessous, $F' < F$ ($-0.012389381$ en
$F = 0.30$). La bissection donne un seuil mesuré de `0.500000000`, et
$F'(1) = 1.000000000`.

Cinq itérations à partir de $F = 0.55$ mènent à `0.560344828`, `0.572842640`,
`0.587909271`, `0.605999438` et `0.627571021`.

Le filtrage local, lui, donne pour chaque mélange de Bell une fraction de
singlet **après** strictement supérieure à $1/2$ : `0.550000000` pour
$w = 0.55$ (contre `0.450000` avant), `0.600000000` pour $w = 0.60` (contre
`0.400000`), `0.700000000` pour $w = 0.70` (contre `0.300000`) et
`0.800000000` pour $w = 0.80` (contre `0.200000`), toujours avec une probabilité
de succès de `0.500000`.

### Interprétation

Trois lectures, dans l'ordre du fil :

- **la fraction de singlet est le bon dénominateur commun.** Pour la famille de
  Werner, la fraction de singlet est $p$ et la borne PPT est $p \le 1/2$ : les
  deux sections se rejoignent sur le même nombre. Le seuil de distillation et
  la frontière de séparabilité sont le même seuil vu par deux outils ;
- **la récurrence est une marche, pas un miracle.** Son point fixe est $F = 1/2$
  : en dessous, on dégrade ; au-dessus, on améliore, mais lentement (de
  `0.55` à `0.63` en cinq itérations). La distillation a un coût : elle exige
  de nombreuses copies et un taux de succès de l'ordre de `0.5` par itération ;
- **le filtrage local est plus violent.** Il ne dégrade rien : il **conditionne**
  sur un succès et sort une fraction de singlet qui franchit le seuil en une
  seule opération. C'est le mécanisme que l'on préfère en pratique — et c'est
  exactement ce que dit Horodecki et al. 2009 §VII.D.

### Exercice 4 — du bruit au singlet

**Objectif.** Mesurer la fraction de singlet d'un état de Werner, vérifier la
récurrence de Bennett, et appliquer le protocole de filtrage local.

**Indice.** `singlet_fraction`, `bennett_recurrence`, `filtering_distillation`.
Le seuil annoncé par Bennett et al. est $F = 1/2$.

**Etape 1.** Pour l'état de Werner de paramètre $p = 0.4$, donnez la fraction
de singlet et la négativité, et dites si l'état est distillable.

**Etape 2.** Pour $p = 0.4$, calculez $F'$, la fraction de singlet après une
itération de la récurrence, et dites si l'itération améliore l'état.

**Etape 3.** Appliquez le filtrage local à cet état et donnez la fraction de
singlet obtenue ainsi que la probabilité de succès.

**Etape 4.** Recommencez l'étape 3 avec $p = 0.9$ et comparez les deux
résultats. Que se passe-t-il sur la probabilité de succès ?

In [9]:
# Exercice a completer
# Etape 1 : fraction de singlet et negativite de W(0.4)
f_04 = None  # TODO etudiant
neg_04 = None  # TODO etudiant
distillable = None  # TODO etudiant : booleen

# Etape 2 : une iteration de la recurrence
f_prime_04 = None  # TODO etudiant
ameliore = None  # TODO etudiant : booleen

# Etape 3 : filtrage local sur W(0.4)
f_filtre = None  # TODO etudiant
succes = None  # TODO etudiant

# Etape 4 : filtrage local sur W(0.9)
f_filtre_09 = None  # TODO etudiant
succes_09 = None  # TODO etudiant

print("Exercice a completer")

Exercice a completer


## 6. Les canaux qui brisent l'intrication

Un **canal** $\Lambda$ est une application linéaire de l'espace des opérateurs
densité sur lui-même, qui préserve la trace. Il est **briseur d'intrication**
(*entanglement-breaking*, EB) s'il transforme **tout** état intriqué en état
séparable : après lui, plus rien n'est intriqué.

Le critère de **Choi** (Horodecki, Shor, Ruskai 2003) dit qu'un canal est EB
si et seulement si sa matrice de Choi
$$J_\Lambda = \sum_{i,k} |i\rangle\langle k| \otimes \Lambda(|i\rangle\langle k|)$$
est **séparable**. Et comme la matrice de Choi d'un canal est un état bipartite
$2 \otimes 2$ quand le canal est un qubit, on peut décider le problème par le
critère PPT : pour un canal de qubit, **EB $\iff$ PPT(Choi)**.

On mesure ce critère sur le **canal dépolarisant**
$$\Lambda_p(X) = p\,X + (1-p)\,\mathrm{Tr}(X)\,\tfrac{I}{d},$$
dont la matrice de Choi vaut $J = p|\Omega\rangle\langle\Omega| +
(1-p)\tfrac{I}{d}$ où $|\Omega\rangle = \tfrac{1}{\sqrt{d}}\sum_i |ii\rangle$.
Le seuil annoncé par la littérature est $p \le 1/(d+1)$.

On mesure ensuite ce qu'on appelle l'**indice de brisage** : le nombre
d'itérations du canal qu'il faut pour qu'il devienne briseur. Pour la famille
dépolarisante, itérer $n$ fois revient à appliquer $\Lambda_{p^n}$, donc la
condition est $p^n \le 1/(d+1)$. C'est l'indice **par itération** ; l'indice
par puissance tensorielle (Lami et Giovannetti 2015) est une autre notion, que
ce carnet ne calcule pas.

In [10]:
def choi(channel, d):
    """Matrice de Choi non normalisee : J = sum_ik |i><k| (x) E(|i><k|)."""
    j = np.zeros((d * d, d * d), dtype=complex)
    for i in range(d):
        for k in range(d):
            e_ik = np.zeros((d, d), dtype=complex)
            e_ik[i, k] = 1.0
            j += np.kron(e_ik, channel(e_ik))
    return j


def depolarizing(p, d):
    """Canal depolarisant trace-preserving : L(X) = p X + (1-p) Tr(X) I/d."""
    def channel(x):
        return p * x + (1 - p) * np.trace(x) * np.eye(d, dtype=complex) / d
    return channel


def eb_iteration_index(p, d, max_iterations=500):
    """Nombre minimal d'iterations avant que le canal depolarisant soit EB.

    Iterer n fois le depolarisant revient au depolarisant de parametre p^n ;
    le canal est EB ssi p^n <= 1/(d+1).
    """
    n = 0
    while p ** n > 1.0 / (d + 1) and n < max_iterations:
        n += 1
    return n


print("--- la matrice de Choi du canal identite ---")
j_ident = choi(lambda x: x, 2)
print("  valeurs propres de J(identite) =", eigenvalues(j_ident))
print("  PPT(Choi) =", is_ppt(j_ident, 2, 2))

print()
print("--- seuil EB du canal depolarisant, mesure par bissection sur le Choi ---")
for d in (2, 3):
    borne = ppt_bound(lambda p: choi(depolarizing(p, d), d), d)
    print(f"  d = {d} : seuil EB mesure = {borne:.9f}   "
          f"(theorie : 1/(d+1) = {1.0 / (d + 1):.9f})")
    for p in (0.0, 0.2, 1.0 / (d + 1), 0.4, 0.6, 1.0):
        j = choi(depolarizing(p, d), d)
        print(f"    p = {p:.2f} : PPT(Choi) = {is_ppt(j, d, d)}   "
              f"valeur propre minimale = {eigenvalues(partial_transpose(j, d, d)).min():+.6f}")

print()
print("--- indice de brisage par iteration ---")
for d in (2, 3):
    for p in (0.9, 0.7, 0.5):
        n = eb_iteration_index(p, d)
        avant = choi(depolarizing(p ** (n - 1), d), d) if n > 0 else None
        print(f"  d = {d} p = {p:.2f} : n = {n}   p^n = {p ** n:.9f}   "
              f"1/(d+1) = {1.0 / (d + 1):.9f}   "
              f"PPT au rang n = {is_ppt(choi(depolarizing(p ** n, d), d), d, d)}   "
              f"PPT au rang n-1 = "
              f"{is_ppt(avant, d, d) if avant is not None else 'n/a'}")

--- la matrice de Choi du canal identite ---
  valeurs propres de J(identite) = [0. 0. 0. 2.]
  PPT(Choi) = False

--- seuil EB du canal depolarisant, mesure par bissection sur le Choi ---
  d = 2 : seuil EB mesure = 0.333333334   (theorie : 1/(d+1) = 0.333333333)
    p = 0.00 : PPT(Choi) = True   valeur propre minimale = +0.500000
    p = 0.20 : PPT(Choi) = True   valeur propre minimale = +0.200000
    p = 0.33 : PPT(Choi) = True   valeur propre minimale = +0.000000
    p = 0.40 : PPT(Choi) = False   valeur propre minimale = -0.100000
    p = 0.60 : PPT(Choi) = False   valeur propre minimale = -0.400000
    p = 1.00 : PPT(Choi) = False   valeur propre minimale = -1.000000
  d = 3 : seuil EB mesure = 0.250000001   (theorie : 1/(d+1) = 0.250000000)
    p = 0.00 : PPT(Choi) = True   valeur propre minimale = +0.333333
    p = 0.20 : PPT(Choi) = True   valeur propre minimale = +0.066667
    p = 0.25 : PPT(Choi) = True   valeur propre minimale = +0.000000
    p = 0.40 : PPT(Choi) = False   

### Lecture du résultat

La matrice de Choi du canal identité a pour valeurs propres `[0. 0. 0. 2.]` :
c'est bien le projecteur non normalisé $|\Omega\rangle\langle\Omega\rangle$ (avec
$|\Omega\rangle = |00\rangle + |11\rangle$), un état pur maximalement
entrelacé. Elle n'est **pas** PPT — c'est la mesure de contrôle attendue : le
canal identité ne brise rien, il conserve l'intrication.

Le seuil EB mesuré par bissection sur la Choi du canal dépolarisant vaut
`0.333333334` pour $d = 2$ et `0.250000001` pour $d = 3$ — soit exactement
$1/(d+1)$ dans les deux cas. Le long du balayage, la Choi passe de PPT
(`valeur propre minimale` à `+0.200000` en $p = 0.2$) à non PPT
(`-0.100000` en $p = 0.4$) ; en $p = 1/3$ exactement, la valeur propre
minimale est nulle, au signe près.

L'indice de brisage par itération vaut `11` pour $d = 2$ et $p = 0.9`
($p^{11} = 0.313810$ contre $1/3 = 0.333333$), `4` pour $p = 0.7`, `2` pour
$p = 0.5$ ; et `14`, `4`, `2` respectivement pour $d = 3$. À chaque fois, la
Choi est PPT au rang $n$ et ne l'est plus au rang $n - 1$.

### Interprétation

La Choi du canal identité donne la mesure de contrôle : sans canal, rien ne
brise rien, et la Choi est un projecteur pur maximalement entrelacé — dont la
transposition partielle a une valeur propre négative, donc non PPT. C'est
exactement ce qu'on attend d'un canal qui n'est pas briseur.

Le seuil `$1/(d+1)$` est **le même nombre** que celui de la section 4 pour la
famille isotrope. Ce n'est pas une coïncidence : les deux seuils sont la
condition que la Choi du canal — ou l'état isotrope — ait une valeur propre
nulle exactement à la frontière de séparabilité. Un canal dépolarisant à
$p = 1/3$ est donc le **dual** d'un état isotrope à $F = 1/3$ : l'un brise
toute intrication, l'autre est à la frontière de la séparabilité.

L'indice de brisage donne la mesure physique du bruit : un canal de qualité
$p = 0.9` — très proche de l'identité — demande `11` itérations pour briser
l'intrication, alors qu'un canal de qualité `0.5` en demande `2`. C'est la
forme quantitative de « le bruit se cumule, mais lentement ».

### Exercice 5 — quand le canal devient briseur

**Objectif.** Calculer la Choi d'un canal, décider s'il brise l'intrication, et
mesurer son indice de brisage.

**Indice.** `choi`, `depolarizing`, `is_ppt` sur la Choi, `eb_iteration_index`.

**Etape 1.** Calculez la Choi du canal dépolarisant de paramètre $p = 0.2$ en
dimension 2, et ses valeurs propres.

**Etape 2.** Ce canal est-il briseur d'intrication ? Justifiez par le critère
de Choi.

**Etape 3.** Calculez l'indice de brisage par itération du canal
dépolarisant de paramètre $p = 0.6$ en dimension 2.

**Etape 4.** Vérifiez par le calcul de la Choi que le rang $n$ est bien le
premier rang PPT, et que le rang $n - 1$ ne l'est pas.

In [11]:
# Exercice a completer
# Etape 1 : Choi du depolarisant p = 0.2 en d = 2
j = None  # TODO etudiant : matrice 4 x 4
valeurs_propres = None  # TODO etudiant

# Etape 2 : ce canal brise-t-il l'intrication ?
est_briseur = None  # TODO etudiant : booleen
justification = None  # TODO etudiant : une phrase en commentaire

# Etape 3 : indice de brisage par iteration pour p = 0.6
n = None  # TODO etudiant

# Etape 4 : PPT au rang n et non PPT au rang n-1
ppt_n = None  # TODO etudiant : booleen
ppt_n_moins_1 = None  # TODO etudiant : booleen

print("Exercice a completer")

Exercice a completer


## 7. Synthèse — de l'état de Bell au canal briseur

Les six sections s'enchaînent sur un même objet : un état de Werner bruité,
$\rho = W(p)$, dont on suit toute la chaîne de Properties. On prend ici
$p = 0.6$ — un état manifestement intriqué, mais bruité.

La chaîne se lit dans cet ordre :

1. **est-il PPT ?** non, donc il est intriqué (mesuré) ;
2. **quelle est sa fraction de singlet ?** $F = p$ (mesuré) ;
3. **le filtrage local améliore-t-il l'état ?** et avec quel coût ? (mesuré) ;
4. **que fait la récurrence de Bennett ?** (mesuré) ;
5. **quel canal brise ce genre d'intrication ?** et au bout de combien
   d'itérations ? (mesuré).

In [12]:
print("=== chaine complete sur l'etat de Werner W(0.6) ===")
rho = werner_state(0.6, 2)

print(f"1. PPT = {is_ppt(rho)}   negativite = {negativity(rho):.6f}")
print(f"2. fraction de singlet F = {singlet_fraction(rho):.9f}")
f_apres, succes, minimum = filtering_distillation(rho)
print(f"3. filtrage local : F_apres = {f_apres:.9f}"
      f"   probabilite de succes = {succes:.6f}"
      f"   min eig(rho^Gamma) = {minimum:+.6f}")
print(f"4. recurrence de Bennett : F' = {bennett_recurrence(singlet_fraction(rho)):.9f}")
print(f"5. canal depolarisant de qualite 0.9 : Choi PPT = "
      f"{is_ppt(choi(depolarizing(0.9, 2), 2), 2, 2)}"
      f"   indice de brisage par iteration = {eb_iteration_index(0.9, 2)}")

print()
print("=== le meme etat de Bell que dans la section 5, pour comparaison ===")
rho_bell_melange = mixed_state([0.55, 0.45],
                                [projector(bell_state("Phi+")),
                                 projector(bell_state("Psi-"))])
print(f"F_avant = {singlet_fraction(rho_bell_melange):.6f}"
      f"   PPT = {is_ppt(rho_bell_melange)}")
f_filtre, succes_filtre, _ = filtering_distillation(rho_bell_melange)
print(f"F_apres filtrage = {f_filtre:.9f}"
      f"   probabilite de succes = {succes_filtre:.6f}")

=== chaine complete sur l'etat de Werner W(0.6) ===
1. PPT = False   negativite = 0.100000
2. fraction de singlet F = 0.600000000
3. filtrage local : F_apres = 0.600000000   probabilite de succes = 0.500000   min eig(rho^Gamma) = -0.100000
4. recurrence de Bennett : F' = 0.620437956
5. canal depolarisant de qualite 0.9 : Choi PPT = False   indice de brisage par iteration = 11

=== le meme etat de Bell que dans la section 5, pour comparaison ===
F_avant = 0.450000   PPT = False
F_apres filtrage = 0.550000000   probabilite de succes = 0.500000


### Lecture du résultat

Sur $\rho = W(0.6)$ : le critère PPT répond `False` et la négativité vaut
`0.100000` — l'état est intriqué. Sa fraction de singlet est `0.600000000`, ce qui
est *exactement* $p$, comme annoncé à la section 5. Le filtrage local rend
`0.600000000` avec une probabilité de succès de `0.500000` et une valeur propre
minimale de `-0.100000` ; la récurrence de Bennett donne `0.620437956`.

Le canal dépolarisant de qualité `0.9` n'est pas briseur d'intrication (Choi non
PPT) et son indice de brisage par itération est `11`. Le mélange de Bell de la
section 5, lui, part de `F = 0.450000` et sort du filtrage à `0.550000000` avec
une probabilité de succès de `0.500000`.

### Interprétation

La chaîne met en évidence une nuance qu'il faut ne pas rater : **le filtrage
local n'améliore pas tout le monde**. Sur $W(0.6)$, il rend $F = 0.6$ — il ne
change rien à la fraction de singlet. La raison est que l'état est *déjà* au
-dessus du seuil $1/2$ : le filtre associé au vecteur propre négatif de
$\rho^\Gamma$ est ici un filtre trivial, et il ne reste rien à gagner. Sur le
mélange de Bell, en revanche, où $F = 0.45 < 1/2$, le même protocole fait
franchir le seuil en une opération, au prix d'une probabilité de succès de
`0.5`.

Autrement dit : **la distillation coûte, et son rendement dépend de l'état de
départ**. C'est le sens de l'indice de brisage — un canal de qualité `0.9` peut
être appliqué onze fois avant de détruire toute intrication.

### Exercice 6 — refaire la chaîne sur un autre état

**Objectif.** Reprendre la chaîne complète sur l'état de Werner de paramètre
$p = 0.55$ et sur l'état de Bell $|\Phi^+\rangle$, et comparer.

**Indice.** Reutilisez `werner_state`, `is_ppt`, `negativity`,
`singlet_fraction`, `filtering_distillation`, `bennett_recurrence`.

**Etape 1.** Pour $\rho = W(0.55)$ : PPT, négativité, fraction de singlet.

**Etape 2.** Filtrez $\rho$ : fraction de singlet obtenue, probabilité de
succès. Cette fraction franchit-elle le seuil de $1/2$ ?

**Etape 3.** Pour l'état pur $|\Phi^+\rangle$ : donnez sa fraction de singlet, et
expliquez pourquoi il n'a pas besoin d'être distillé.

**Etape 4.** Concluez en une phrase : pour quels paramètres $p$ l'état de
Werner est-il distillable ?

In [13]:
# Exercice a completer
# Etape 1 : W(0.55) — PPT, negativite, fraction de singlet
ppt = None  # TODO etudiant : booleen
neg = None  # TODO etudiant
f = None  # TODO etudiant

# Etape 2 : filtrage local de W(0.55)
f_apres = None  # TODO etudiant
succes = None  # TODO etudiant
franchit = None  # TODO etudiant : booleen

# Etape 3 : etat de Bell |Phi+>
f_bell = None  # TODO etudiant

# Etape 4 : pour quels p l'etat de Werner est-il distillable ?
conclusion = None  # TODO etudiant : une phrase en commentaire

print("Exercice a completer")

Exercice a completer


## À retenir

1. **Un état mixte n'est ni intricable ni séparable sans critère.** Le critère
   de la page réduite tranche le cas **pur** (une page réduite pure ⟺ état
   produit) et **échoue** sur le cas mixte : $\tfrac12(|00\rangle\langle 00| +
   |11\rangle\langle 11|)$ a des pages réduites maximally mélangées et est
   pourtant séparable (Mesuré).
2. **Un témoin d'intrication est un outil à sens unique.** Le témoin
   $\langle\Phi^+|\rho|\Phi^+\rangle$ a une borne séparable de `0.5`, atteinte
   exactement par les états produit ; au-dessus, l'état est intriqué ; en
   dessous, le témoin n'affirme **rien** sur la séparabilité (Mesuré).
3. **PPT est nécessaire, et suffisant seulement en basse dimension.** La
   transposition partielle $\rho^\Gamma$ d'un état séparable est toujours
   positive ; l'inverse vaut en $2 \otimes 2$ et $2 \otimes 3$ (Cité :
   Peres 1996, Horodecki 1996), et fails en $3 \otimes 3$ et au-delà, où
   existent des états PPT **intriqués**, donc **non distillables** (Cité :
   Horodecki 1997, Horodecki 2009 §VI.B.7).
4. **Les bornes se mesurent.** La borne PPT de la famille de Werner vaut
   `0.500000001` pour $d = 2$, `0.500000001` pour $d = 3$ et `0.500000002`
   pour $d = 4$ ; celle de la famille isotrope vaut `0.333333335`, `0.250000002`
   et `0.200000003` — soit $1/2$ et $1/(d+1)$ (Mesuré).
5. **La fraction de singlet est le dénominateur commun.** Pour l'état de Werner
   en dimension 2, $F = p$ **exactement** (Mesuré) : la frontière de
   séparabilité $p \le 1/2$ et le seuil de distillation $F > 1/2$ sont le même
   nombre.
6. **La récurrence de Bennett a pour point fixe $F = 1/2$.** Mesurée par
   bissection, l'amélioration $F' > F$ commence à `0.500000000` ; à partir de
   $F = 0.55$, cinq itérations mènent à `0.627571021` (Mesuré).
7. **Le filtrage local franchit le seuil en une opération, avec un coût.**
   Un mélange de Bell passe de $F = 0.45$ à `0.550000000` avec une probabilité
   de succès de `0.500000` (Mesuré ; Horodecki 2009 §VII.D, eq. 106-107).
8. **Un canal brise l'intrication quand sa Choi est séparable.** Pour un
   qubit, EB $\iff$ PPT(Choi) (Cité : Horodecki-Shor-Ruskai 2003) ; le canal
   dépolarisant est EB ssi $p \le 1/(d+1)$, seuil mesuré à `0.333333334` et
   `0.250000001` ; son indice de brisage par itération est le plus petit $n$
   tel que $p^n \le 1/(d+1)$, soit `11` pour $p = 0.9$ en dimension 2
   (Mesuré).
9. **Reste ouvert.** La question « **tous les états NPT sont-ils distillables ?** »
   est ouverte depuis la découverte de l'intrication liée (Cité : Horodecki
   2009 §VII.H) ; elle équivaut à la question pour les états de Werner NPT.

## Vérifiez votre compréhension

**1. Pourquoi la page réduite ne suffit-elle pas à conclure sur un état
mixte ?**

<details>
<summary>Réponse</summary>

Pour un état **pur**, une page réduite pure équivaut à un état produit : le
critère est complet. Pour un état **mixte**, la page réduite peut être
maximalement mélangée alors que l'état est séparable — c'est le cas de
$\tfrac12(|00\rangle\langle 00| + |11\rangle\langle 11|)$, dont la section 2
mesure une page réduite de pureté `0.5`. La condition « page réduite mélangée »
est alors **nécessaire** (tout état séparable a des pages réduites mixtes),
mais **pas suffisante**.

</details>

**2. Que vaut le témoin $\langle\Phi^+|\rho|\Phi^+\rangle$ sur la famille de
Werner, et que prouve un témoin qui se tait ?**

<details>
<summary>Réponse</summary>

Sur $W(p)$, le témoin vaut $(1-p)/3$, soit au plus `0.333333` — toujours en
dessous de la borne séparable `0.5`. Le témoin est donc **muet** sur
l'intrication de toute la famille, alors que ces états sont intriqués dès que
$p > 1/2$ (Mesuré). Un témoin qui se tait ne prouve **rien** : il indique
seulement que cet outil-là ne voit pas cette intrication. Il faut alors un
critère qui porte sur toutes les séparations — le PPT.

</details>

**3. Pourquoi le critère PPT suffit-il en $2 \otimes 2$ et $2 \otimes 3$, et
pas au-delà ?**

<details>
<summary>Réponse</summary>

C'est un résultat **cité** (Horodecki et al. 1996), pas une mesure de ce carnet :
la structure même de l'état dans ces dimensions impose que PPT implique
séparable. À partir de $3 \otimes 3$, il existe des états PPT **intriqués**
(Cité : Horodecki 1997) : le critère devient seulement nécessaire, et ces
états sont **non distillables** — c'est l'intrication liée. Le carnet mesure
les bornes des familles classiques, mais ne peut pas, avec PPT seul, produire
un exemple d'état PPT intriqué : il le dit explicitement.

</details>

**4. Quel est le seuil de distillabilité d'un état de Werner, et comment le
lit-on dans le code ?**

<details>
<summary>Réponse</summary>

En dimension 2, la fraction de singlet de $W(p)$ vaut exactement $p$ (Mesuré).
Le seuil de distillation est donc $p = 1/2$, c'est-à-dire $F = 1/2$. Le code
le mesure indépendamment par bissection sur la condition $F' > F$ de la
récurrence de Bennett : la mesure donne `0.500000000`, et $F'(1/2) = 0.5$ est
le point fixe de la récurrence (Mesuré).

</details>

**5. Pourquoi un canal de qualité `0.9` demande-t-il `11` itérations pour
briser l'intrication ?**

<details>
<summary>Réponse</summary>

Itérer $n$ fois le canal dépolarisant revient à appliquer $\Lambda_{p^n}$, et le
canal est briseur ssi sa Choi est séparable, ce qui pour cette famille
equivaut à $p^n \le 1/(d+1)$. En dimension 2 : $0.9^{10} = 0.348678 > 1/3$
(pas encore briseur) et $0.9^{11} = 0.313810 \le 1/3$ (brisable) — d'où `11`.
Le code vérifie la Choi aux deux rangs (Mesuré).

</details>

**6. Quelle question reste ouverte, et à quoi se rattache-t-elle ?**

<details>
<summary>Réponse</summary>

La distillabilité des états **NPT** : « tous les états NPT sont-ils
distillables ? » (Cité : Horodecki 2009 §VII.H). L'équivalence est Established
avec la question posée sur les seuls états de Werner NPT : tout état intriqué
peut être ramené, par filtrage puis twirling, à un état de Werner
intriqué. Une réponse positive exclurait toute intrication liée au sens NPT.

</details>

## Pour aller plus loin

- **[Lean-13b — CHSH et Tsirelson](../../SymbolicAI/Lean/Lean-13b-CHSH-Tsirelson-Native.ipynb)** —
  la non-localité **formellement prouvée** : le même objet (des corrélations
  quantiques qui dépassent la borne classique) traité par un organe de preuve
  plutôt que par du calcul numérique. À lire après la section 3 : le témoin
  d'intrication y a un analogue non-local.
- **[Constructeur-01 — les lois comme tâches](../Constructeur/Constructeur-01-Lois-Comme-Taches-Python.ipynb)** —
  contient une fonction `negativite(rho)` : l'organe le plus proche de ce
  carnet dans le dépôt, à comparer avec la section 4.
- **[Chaînes-01 — l'axiome d'intrication](../Chaines/Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb)** —
  l'autre usage du mot « intrication » : un **lien** entre deux systèmes dans
  un état **pur** maximalement entrelacé, soumis à un plafond de lisibilité.
  Les deux carnets ne parlent pas du même objet ; les lire de suite lève la
  confusion.
- **Les sources primaires** — les neuf références ci-dessous sont archivées au
  gisement partagé et sont citées avec leur chemin exact.

## Références

Toutes les références sont archivées au gisement partagé
`G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\` et citées avec leur chemin
complet. Aucun PDF n'est versionné dans ce dépôt.

**Séparabilité et critère PPT**

- A. Peres, *Separability Criterion for Density Matrices* (1996), arXiv:9604005 —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\1996 - Peres - Separability Criterion for Density Matrices (arXiv 9604005).pdf`
- M. Horodecki, P. Horodecki, R. Horodecki, *Separability of Mixed States — Necessary and Sufficient Conditions* (1996), arXiv:9605038 —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\1996 - Horodecki et al - Separability of Mixed States - Necessary and Sufficient Conditions (arXiv 9605038).pdf`
- M. Horodecki, P. Horodecki, R. Horodecki, *Quantum Entanglement* (2009), arXiv:0702225 — §IX (familles de Werner et isotrope, eq. 71-72), §VI.B.7 (intrication liée), §VII.D (filtrage local, eq. 106-107), §VII.H (question NPT) —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2009 - Horodecki et al - Quantum Entanglement (arXiv 0702225).pdf`

**Distillation**

- C. H. Bennett, D. P. DiVincenzo, P. W. Shor, J. A. Smolin, *Purification of Noisy Entanglement and Faithful Teleportation via Noisy Channels* (1996), arXiv:9511027 — la fraction de singlet et la récurrence eq. (7) —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\1996 - Bennett et al - Purification of Noisy Entanglement and Faithful Teleportation via Noisy Channels (arXiv 9511027).pdf`
- M. Horodecki, P. Horodecki, R. Horodecki, *General Teleportation Channel, Singlet Fraction and Quasi-Distillation* (1999), arXiv:9807091 —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\1999 - Horodecki et al - General Teleportation Channel, Singlet Fraction and Quasi-Distillation (arXiv 9807091).pdf`

**Canaux brisant l'intrication**

- M. Horodecki, P. Shor, M. B. Ruskai, *Entanglement Breaking Channels* (2003), arXiv:0302031 — le critère de Choi —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2003 - Horodecki Shor Ruskai - Entanglement Breaking Channels (arXiv 0302031).pdf`
- I. Lami, V. Giovannetti, *Entanglement-Breaking Indices* (2015), arXiv:1411.2517 — l'indice par puissance tensorielle, distinct de l'indice par itération calculé ici —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2015 - Lami Giovannetti - Entanglement-Breaking Indices (arXiv 1411.2517).pdf`
- M. M. Wilde, *From Classical to Quantum Shannon Theory* (2019), arXiv:1106.1445 — le canal briseur et l'équivalence mesure-et-prépare —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2019 - Wilde - From Classical to Quantum Shannon Theory (arXiv 1106.1445).pdf`

**Inégalités de Bell (usage dans la sous-série, contexte)**

- R. F. Werner et M. M. Wolf, *All Multipartite Bell Correlation Inequalities for Two Dichotomic Observables per Site* (2001), arXiv:0102024 — les états de Werner-Wolf et les inégalités de Bell multi-parties ; **cité ici pour le contexte**, pas pour la séparabilité —
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2001 - Werner Wolf - All Multipartite Bell Correlation Inequalities for Two Dichotomic Observables per Site (arXiv 0102024).pdf`